In [1]:
import numpy as np
import pandas as pd
import torch

# NOTES: Compared to the source code chosen, there's gonna be differences that I haven't added into this notebook but the general ideas behind what I made is the same: i.e. the biochemical feature vectors + radius-based sequencing

In [2]:
AA_IDS = [
    'A','C','D','E','F','G','H','I','K','L','M','N','P','Q','R','S','T','V','W','Y','VOID',
]

# molecular weights are in kilodaltons
MOLECULAR_WEIGHTS = {
    'A': 0.09,
    'C': 0.12,
    'D': 0.13,
    'E': 0.15,
    'F': 0.17,
    'G': 0.08,
    'H': 0.16,
    'I': 0.13,
    'K': 0.15,
    'L': 0.13,
    'M': 0.15,
    'N': 0.13,
    'P': 0.12,
    'Q': 0.15,
    'R': 0.17,
    'S': 0.11,
    'T': 0.12,
    'V': 0.12,
    'W': 0.20,
    'Y': 0.18,
    'VOID': 0,
}

# net charges at physiological pH ~7.4
NET_CHARGES = {
    'A': 0,
    'C': 0,
    'D': -1,
    'E': -1,
    'F': 0,
    'G': 0,
    'H': 0,
    'I': 0,
    'K': +1,
    'L': 0,
    'M': 0,
    'N': 0,
    'P': 0,
    'Q': 0,
    'R': +1,
    'S': 0,
    'T': 0,
    'V': 0,
    'W': 0,
    'Y': 0,
    'VOID': 0,
}

# isoelectric point pHs from peptide web
# NOTE: will retrain model since I was using old textbook values with one outdated value -> shouldn't affect model accuracy much
ISOELECTRIC_PTS = {
    'A': 6.02,
    'C': 5.02,
    'D': 2.98,
    'E': 3.22,
    'F': 5.48,
    'G': 5.97,
    'H': 7.59,
    'I': 5.98,
    'K': 9.47,
    'L': 5.98,
    'M': 5.75,
    'N': 5.41,
    'P': 6.30,
    'Q': 5.65,
    'R': 10.76,
    'S': 5.68,
    'T': 5.60,
    'V': 5.97,
    'W': 5.94,
    'Y': 5.66,
    'VOID': 0,
}

# hydrophobicity levels - pH 7
# +ve values and up = hydrophobic
# around 0 = neutral
# -ve values = hydrophilic
HYDROPHOBICITY_IDXS = {
    'A': 41,
    'C': 49,
    'D': -55,
    'E': -31,
    'F': 100,
    'G': 0,
    'H': 8,
    'I': 100,
    'K': -23,
    'L': 97,
    'M': 74,
    'N': -28,
    'P': -46,
    'Q': -10,
    'R': -14,
    'S': -5,
    'T': 13,
    'V': 76,
    'W': 97,
    'Y': 63,
    'VOID': 0,
}


# individual half lives
HALF_LIFE = {
    'A': 4.4,
    'C': 1.2,
    'D': 1.1,
    'E': 1,
    'F': 1,
    'G': 30,
    'H': 3.5,
    'I': 20,
    'K': 1.3,
    'L': 5.5,
    'M': 30,
    'N': 1.4,
    'P': 21,
    'Q': 0.8,
    'R': 1,
    'S': 1.9,
    'T': 7.2,
    'V': 100,
    'W': 2.8,
    'Y': 2.8,
    'VOID': 0,
}


# pseudo molecular fingerprint accounting for biochemical / functional group standout features
# check function_fp_scheme.txt for more understanding
FUNCTIONAL_FP = {
    "A": [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    "G": [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    "D": [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    "E": [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    "K": [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    "R": [0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    "H": [0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    "N": [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
    "Q": [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
    "C": [0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0],
    "M": [0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0],
    "F": [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0],
    "Y": [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0],
    "W": [0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0],
    "L": [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0],
    "I": [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0],
    "V": [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0],
    "S": [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0],
    "T": [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0],
    "P": [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
    "VOID": [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
}

In [70]:
import re
def parse_3d_col(s):
    arrays = re.findall(r"array\(\[(.*?)\]\)", s)
    return [np.array([float(x) for x in a.split(',')]) for a in arrays]

In [4]:
# do a bit of data exploration
import ast
import re
def parse_3d_col(s):
    arrays = re.findall(r"array\(\[(.*?)\]\)", s)
    return [np.array([float(x) for x in a.split(',')]) for a in arrays]
AA3D_df = pd.read_csv("notebook_database/AA3D_df.csv")
AA3D_df["AA_IDs"] = AA3D_df['AA_IDs'].apply(ast.literal_eval)
AA3D_df["3d_struct"] = AA3D_df["3d_struct"].apply(parse_3d_col)

In [5]:

all_vectors = np.concatenate(AA3D_df["3d_struct"].values)  # shape: (total_atoms, 3)
norms = np.linalg.norm(all_vectors, axis=1)

print(f"max radius: {norms.max():.4f} Å")
print(f"min radius: {norms.min():.4f} Å")

# max and min raw A directions were 25 and -27

max radius: 28.4715 Å
min radius: 0.5505 Å


In [6]:
class NAAnoEnc:
    """Run this everytime we need a new set of embeddings"""
    def __init__(self, verbose=False):
        self.nAAno_emb = {}   # basically just make it easier to embed down the line
        self.verbose = verbose

    def initialize(self):
        self.nAAno_emb = {aa_id: get_embedding(aa_id) for aa_id in AA_IDS}

        if self.verbose:
            print("NAAnoEnc initialized")
        return True

    def get_aa_id(self, emb_vector):
        aa_id = None
        for code, key in self.nAAno_emb.items():
            if key == emb_vector:
                aa_id = code
                break

        if aa_id is None:
            raise ValueError(f"Embedding vector {emb_vector} presented does not exist")

        return aa_id


def get_embedding(aa_id: str):   # this isn't an embedding this is a feature extraction tool but I'm just gonna leave it as a learning point for myself
    """
    use this in these two scenarios:
    \n    - generating embeddings after updating feature vector
    \n    - inference
    :param aa_id: single letter amino acid reference code
    :returns: feature vector representing a given (valid) amino acid
    """
    # sanity check
    if aa_id not in AA_IDS:
        raise ValueError(f"{aa_id} not in valid AA ID list")

    # embedding scheme, MAKE SURE TO UPDATE THIS IF YOU EVER UPDATE NAANOLIBRARY
    embedding = [
        MOLECULAR_WEIGHTS[aa_id],
        NET_CHARGES[aa_id],
        ISOELECTRIC_PTS[aa_id],
        HYDROPHOBICITY_IDXS[aa_id],
        HALF_LIFE[aa_id],
    ]
    embedding += FUNCTIONAL_FP[aa_id]
    return embedding


def encoder_check():
    encoder = NAAnoEnc(verbose=True)
    encoder.initialize()
    for aa_code, aa_vect in encoder.nAAno_emb.items():
        print(f"{aa_code} -- {aa_vect}")

    # check decoder and encoder
    for aa in AA_IDS:
        aa_str = aa
        aa_emb = get_embedding(aa)
        if aa_str == encoder.get_aa_id(aa_emb):
            print(f"{aa_str}: str <-> vect aligned")

encoder_check()  # note: all good

NAAnoEnc initialized
A -- [0.09, 0, 6.02, 41, 4.4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
C -- [0.12, 0, 5.02, 49, 1.2, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0]
D -- [0.13, -1, 2.98, -55, 1.1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
E -- [0.15, -1, 3.22, -31, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
F -- [0.17, 0, 5.48, 100, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0]
G -- [0.08, 0, 5.97, 0, 30, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
H -- [0.16, 0, 7.59, 8, 3.5, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]
I -- [0.13, 0, 5.98, 100, 20, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0]
K -- [0.15, 1, 9.47, -23, 1.3, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
L -- [0.13, 0, 5.98, 97, 5.5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0]
M -- [0.15, 0, 5.75, 74, 30, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0]
N -- [0.13, 0, 5.41, -28, 1.4, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0]
P -- [0.12, 0, 6.3, -46, 21, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]
Q -- [0.15, 0, 5.65, -10, 0.8, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0]
R -- [0.17, 1,

In [15]:
class RAAdialSeeker:

    def __init__(self, resolution, verbose=False):
        self.resolution = resolution
        self.verbose = verbose

        self.angstrom_lim = 33   # maximum angstrom range found + some extra for context enrichment
                                 # can always edit this later on
        self.angstrom_inc = float(33 / resolution)
        self.threshold = float(1 / resolution)  # standardize how we determine radial sequences
        #                           smallest distance is the base increment, not 0
        self.radius_levels = torch.arange(self.angstrom_inc, self.angstrom_lim + self.angstrom_inc, step=self.angstrom_inc)

        self.coord_layer = None # [1, 3] -> 3D vector

    def init_spAAtial(self):
        self._make_coord_layer()

    def _make_coord_layer(self):
        self.coord_layer = torch.randn(size=(1, 3))

    def radial_sequence(self, aa_seq, vect_seq):
        radial_seq = {}  # lookup table
        seen = []

        # sanity
        if len(aa_seq) != len(vect_seq):
            raise ValueError(f"string and vector sequences of {aa_seq} are different lengths")

        # iterate up resolution increments, if a molecule's coordinate is within like 1/resolution of an angstrom, append it's info
        # its radius is now the unique ID, so if we stumble on it again its not duplicated
        for level in self.radius_levels:
            radial_seq[level] = []
            for i in range(len(aa_seq)):
                num_id = (np.sqrt(sum(vect_seq[i] ** 2)), i)
                if self._dist_check(vect_seq[i], level) and (num_id not in seen):
                    radial_seq[level].append([aa_seq[i], vect_seq[i]])
                    seen.append(num_id)

        # create two VOID tokens (0,0,0) on both sides to denote stops and starts
        return [0, 0, 0] + radial_seq[::-1] + [0, 0, 0]  # we want to go from outside inward

    def _dist_check(self, vector, ang_radius):
        if abs(ang_radius - np.sqrt(sum(vector ** 2))) <= self.threshold:
            return True
        else:
            return False

def test_resolution():
    module = RAAdialSeeker(resolution = 100)
    module.init_spAAtial()
    for level in module.radius_levels:
        print(level)

test_resolution()

tensor(0.3300)
tensor(0.6600)
tensor(0.9900)
tensor(1.3200)
tensor(1.6500)
tensor(1.9800)
tensor(2.3100)
tensor(2.6400)
tensor(2.9700)
tensor(3.3000)
tensor(3.6300)
tensor(3.9600)
tensor(4.2900)
tensor(4.6200)
tensor(4.9500)
tensor(5.2800)
tensor(5.6100)
tensor(5.9400)
tensor(6.2700)
tensor(6.6000)
tensor(6.9300)
tensor(7.2600)
tensor(7.5900)
tensor(7.9200)
tensor(8.2500)
tensor(8.5800)
tensor(8.9100)
tensor(9.2400)
tensor(9.5700)
tensor(9.9000)
tensor(10.2300)
tensor(10.5600)
tensor(10.8900)
tensor(11.2200)
tensor(11.5500)
tensor(11.8800)
tensor(12.2100)
tensor(12.5400)
tensor(12.8700)
tensor(13.2000)
tensor(13.5300)
tensor(13.8600)
tensor(14.1900)
tensor(14.5200)
tensor(14.8500)
tensor(15.1800)
tensor(15.5100)
tensor(15.8400)
tensor(16.1700)
tensor(16.5000)
tensor(16.8300)
tensor(17.1600)
tensor(17.4900)
tensor(17.8200)
tensor(18.1500)
tensor(18.4800)
tensor(18.8100)
tensor(19.1400)
tensor(19.4700)
tensor(19.8000)
tensor(20.1300)
tensor(20.4600)
tensor(20.7900)
tensor(21.1200)
tensor

In [9]:
import torch

class Shell:
    """
    Basically a geometric tokenizer
    Performs coordinate discretization for any given 3D vector
    """
    def __init__(self, shell_resolution, max_angstroms, smooth):
        self.shell_resolution = shell_resolution + 1  # one more slot for the 0
        self.max_angstroms = max_angstroms  # make sure this is the same in RAAdialSeeker
        self.shell_increment = max_angstroms / shell_resolution
        self.smooth = smooth

        # EACH AXIS IS STRUCTURED AS SUCH [0, max_angstroms ....... -max_angstroms]
        self.x_axis = torch.zeros(shell_resolution)
        self.y_axis = torch.zeros(shell_resolution)
        self.z_axis = torch.zeros(shell_resolution)


    def vect2idx(self, vector):
        """Converts a torch.tensor of 3D vectors into their nearest shell-resolution cell index"""
        idxs = torch.round(vector / self.shell_increment)
        idxs = torch.clamp(idxs, -self.shell_resolution, self.shell_resolution).long()
        return idxs

    def idx2vect(self, idxs):
        """Converts a torch.tensor of indexes back into their original 3D vectors"""
        return idxs * self.shell_increment


In [11]:
radial_module = RAAdialSeeker(resolution = 100)
shell_module = Shell(shell_resolution=100, max_angstroms=33, smooth=1)
new_3d_df = "radial_sequences.csv"

# NOTE: RADIAL SEQUENCES:
# for each row (radial layer) -> list of lists
# nested list: [0] is the AA_ID -> embed this with NAAnoEnc in training
#              [1] is the 3D vector: no embedding necessary

# I'm going to treat the protein space like a sequence, with wavenet
# Don't focus on the shell layers, just feed and embed the coordinates
# That way the model can still generate proteins on the same "shell"
# If the model samples a void token, end / break
# context would be like 10 positions, so the first aa would have 9 voids behind
# so total "vocab" size would be 100^3 so much more manageable damn its still a lot

# embedding table for the spatial tokens to sample from + predict where the next AA should be placed given its context and SMILES